### Structured output : pydantic , typedDict

In [3]:
import os 
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [21]:
model = init_chat_model("groq:qwen/qwen3.8-27b",max_tokens="1000")

In [5]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int=Field(description="this year the movie was released")
    director:str=Field(description="the one who directed the movie")
    rating:float=Field(description="the movie's imdb rating out of 10")
    

In [ ]:
model_with_structure = model.with_structured_output(Movie, include_raw=True)
model_with_structure

In [13]:
model_with_structure.invoke("provide details about the  movie magadheera ")

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '03w9b694s', 'function': {'arguments': '{"director":"Srikanth Aaddala","rating":7.6,"title":"Magadheera","year":2009}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 355, 'total_tokens': 430, 'completion_time': 0.202155388, 'completion_tokens_details': None, 'prompt_time': 0.025015414, 'prompt_tokens_details': None, 'queue_time': 0.051186223, 'total_time': 0.227170802}, 'model_name': 'qwen/qwen3.8-27b', 'system_fingerprint': 'fp_a1293f40b5', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a17196-6855-7569-835c-c68446cccdd7-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Srikanth Aaddala', 'rating': 7.6, 'title': 'Magadheera', 'year': 2009}, 'id': '03w9b694s', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 355, 'output_tokens': 75, 'total

### Nested Structure

In [16]:
from pydantic import BaseModel,Field

class Actor(BaseModel):
    name: str
    role: str
class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="budget in crores INR")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("provide details about the movie baahubali")

response

MovieDetails(title='Baahubali', year=2015, cast=[Actor(name='Prabhas', role='Baahubali'), Actor(name='Rana Daggubati', role='Amarendra Baahubali'), Actor(name='Anushka Shetty', role='Devasena'), Actor(name='Ramya Krishnan', role='Avanthika'), Actor(name='Samantha', role='Katyayani'), Actor(name='Tamannaah Bhatia', role='Bhavya'), Actor(name='Nawazuddin Siddiqui', role='Bhallaladeva'), Actor(name='R. Madhavan', role='Kattappa'), Actor(name='Sridevi', role='Sivagami'), Actor(name='Hema Malini', role='Ishana'), Actor(name='Amitabh Bachchan', role='Vyaghra Baahubali')], genres=['Action', 'Adventure', 'Drama', 'Fantasy'], budget=86.0)

### typeddict

In [15]:
from typing_extensions import TypedDict,Annotated

class Actor(TypedDict):
    name: str
    role: str
class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="budget in crores INR")
    
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("provide details about the movie baahubali")

response

{'budget': 180000000,
 'cast': [{'name': 'Prabhas', 'role': 'Amarendra Baahubali'},
  {'name': 'Rana Daggubati', 'role': 'Bhallaladeva'},
  {'name': 'Anushka Shetty', 'role': 'Devasena'},
  {'name': 'Rammachandran', 'role': 'Bhima'},
  {'name': 'Samantha Ruth Prabhu', 'role': 'Avanthika'}],
 'genres': ['Action', 'Adventure', 'Drama'],
 'title': 'Baahubali: The Beginning',
 'year': 2015}

In [26]:
model.profile

### DataClasses

In [29]:
from pydantic import  BaseModel,Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
  """Contact information of a person"""
  name: str = Field(description="name of the person")
  email: str = Field(description="email address of the person")
  mobile: str = Field(description="phone no of the person")

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo 
)

result = agent.invoke({
  "messages": [{"role":"user", "content":"Extract contact info from: naveen , navven@gmail.com, 933381239"}]
})

result["structured_response"]

ContactInfo(name='naveen', email='navven@gmail.com', mobile='933381239')

In [30]:
from typing_extensions import TypedDict,Annotated
from langchain.agents import create_agent

class ContactInfo(TypedDict):
  """Contact information of a person"""
  name: str = Field(description="name of the person")
  email: str = Field(description="email address of the person")
  mobile: str = Field(description="phone no of the person")

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo 
)

result = agent.invoke({
  "messages": [{"role":"user", "content":"Extract contact info from: naveen , navven@gmail.com, 933381239"}]
})

result["structured_response"]

{'name': 'naveen', 'email': 'navven@gmail.com', 'mobile': '933381239'}

In [32]:
## data class


from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
  """Contact information of a person"""
  name: str # name of the person
  email: str # email address of the person
  mobile: str  # phone no of the person

agent = create_agent(
    model="groq:qwen/qwen3.8-27b",
    response_format=ContactInfo 
)

result = agent.invoke({
  "messages": [{"role":"user", "content":"Extract contact info from: naveen , navven@gmail.com, 933381239"}]
})

result["structured_response"]

ContactInfo(name='naveen', email='navven@gmail.com', mobile='933381239')